In [0]:
%sql
--Create new Managed Volume named Landing Under Catalog_s.bronze
CREATE VOLUME catalog_s.bronze.landing
COMMENT 'This is a managed Landing Volume';

In [0]:
%python
#Create new folder input under the volume landing
dbutils.fs.mkdirs("dbfs:/Volumes/catalog_s/bronze/landing/input")

In [0]:
%python
#Copy retail invoice data from databricks datasets
dbutils.fs.cp('databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv',
              '/Volumes/catalog_s/bronze/landing/input')

In [0]:
%python
dbutils.fs.cp('databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-02.csv',
              '/Volumes/catalog_s/bronze/landing/input')

In [0]:
--Create a place holder Table catalog_s.bronze.invoice_cp
CREATE TABLE catalog_s.bronze.invoice_cp;

In [0]:
--Use Copy Into COmmand to load the data into the placeholder table
COPY INTO catalog_s.bronze.invoice_cp
FROM '/Volumes/catalog_s/bronze/landing/input'
FILEFORMAT = CSV
PATTERN = '*.csv'
FORMAT_OPTIONS(
  'mergeSchema' = 'true',
  'header' ='true'
)
COPY_OPTIONS(
  'mergeSchema' = 'true'
)
;


In [0]:
select * from catalog_s.bronze.invoice_cp
--where cast(Quantity as double) < 0

In [0]:
DESCRIBE EXTENDED catalog_s.bronze.invoice_cp;

In [0]:
CREATE TABLE catalog_s.bronze.invoice_cp_alt(
  InvoiceNo string,
  StockCode string,
  Quantity double,
  _invoice_date TIMESTAMP
);

In [0]:
--Use Copy Into Command to load the data into the new placeholder table
COPY INTO catalog_s.bronze.invoice_cp_alt
FROM (
  SELECT InvoiceNo, StockCode, cast(Quantity as double), current_timestamp() as _invoice_date
  from '/Volumes/catalog_s/bronze/landing/input')
FILEFORMAT = CSV
PATTERN = '*.csv'
FORMAT_OPTIONS(
  'mergeSchema' = 'true',
  'header' ='true'
)
;


In [0]:
select * from  catalog_s.bronze.invoice_cp_alt 

In [0]:
%python
dbutils.fs.cp('databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv',
              '/Volumes/catalog_s/bronze/landing/input')

In [0]:
describe history catalog_s.bronze.invoice_cp_alt